# Frontier: PLAN 09 actual overlap neighborhoods

This notebook is the live frontier notebook for PLAN 09. It isolates the missing construction of the overlap neighborhoods $V_j$ for the actual high-escaping chart chains.

The older Step 13 counterexample and weakened-theorem status notebook has been archived at `notebooks/archive/plan08_step13_actual_high_escaping_charts_frontier.ipynb`. The comparison frontier is still recorded separately in `notebooks/plan08_step13_overlap_comparison_frontier.ipynb`.


## Current status

Already formalized in Lean:

- finite chart chains along basin loops,
- the abstract overlap-comparison theorem,
- the open-set bridge from equality of local logarithm branches to trivial monodromy.

What is **not** yet formalized is the geometric input needed by those theorems for the genuine high-escaping chains:

- actual charts $U_j$ covering the high-level image,
- actual open overlap neighborhoods $V_j \subset U_j \cap U_{j+1}$ around the overlap values,
- actual local branches $b_j, \xi_j$ on the $U_j$,
- actual proofs that $b_j = b_{j+1}$ on each $V_j$.

This notebook is a sandbox for the missing PLAN 09 step: construct $V_j$ explicitly in a good special case and organize the proof obligations in the form needed later for the genuine theorem.


## Exact local construction problem

Given a basin loop $\gamma$ and a high escaping level $K$, write
$$
A_K(t) = \text{logSeriesBottcherApprox}\bigl(2, f_2^K(\gamma(t))\bigr).
$$
After choosing a partition
$$
0 = t_0 < t_1 < \cdots < t_{m+1} = 1,
$$
PLAN 09 must produce, for every adjacent pair of charts, an open set
$$
V_j \subset U_j \cap U_{j+1}
$$
with
$$
A_K(t_{j+1}) \in V_j,
$$
and then prove that the neighboring actual logarithm branches agree on $V_j$.

Once that is done, the already-formalized theorem
```lean
ChartChainLocalLogsEventuallyEqAtOverlaps.of_open_eqOn
```
and hence
```lean
BasinLoopChartChain.monodromyProduct_eq_one_of_open_eqOn
```
finish the monodromy comparison automatically.


## Special-case sandbox: a four-chart right-half-plane model

As a sandbox, take the loop
$$
\gamma_r(t) = \frac{1}{2} e^{2 \pi i t}
$$
and the level-$2$ image model
$$
A_2(z) = z^4 + 4 z^2 + 6.
$$
In this example the entire image curve stays in the right half-plane, so one global logarithm exists. That does **not** solve the general theorem, but it gives a clean special case in which we can study the missing geometric step directly:

1. choose explicit disk charts $U_j$ covering the image segments,
2. choose explicit overlap disks $V_j$ around the overlap values,
3. verify $V_j \subset U_j \cap U_{j+1}$,
4. observe that the local logarithm and root branches agree on each $V_j$ because they are restrictions of the same right-half-plane branch.

This is the kind of special-case construction that PLAN 09 should generalize beyond the global-log setting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

r = 0.5
K = 2
n_pts = 2400
theta = np.linspace(0, 2 * np.pi, n_pts)
t = theta / (2 * np.pi)
z_curve = r * np.exp(1j * theta)

def A2(z):
    return z**4 + 4 * z**2 + 6

w_curve = A2(z_curve)
b_curve = np.log(w_curve)
xi_curve = np.exp(b_curve / (2 ** K))

n_sets = 4
colors = ['#4c78a8', '#f58518', '#54a24b', '#e45756']
edges = np.linspace(0, len(w_curve) - 1, n_sets + 1, dtype=int)
intervals = [(edges[j], edges[j + 1]) for j in range(n_sets)]

u_centers = []
u_radii = []
for a, b in intervals:
    seg = w_curve[a:b + 1]
    center = seg.mean()
    radius = np.max(np.abs(seg - center)) + 0.08
    u_centers.append(center)
    u_radii.append(radius)

v_centers = []
v_radii = []
overlap_pads = []
for j in range(n_sets - 1):
    point = w_curve[edges[j + 1]]
    cap_left = u_radii[j] - abs(point - u_centers[j])
    cap_right = u_radii[j + 1] - abs(point - u_centers[j + 1])
    radius = 0.45 * min(cap_left, cap_right)
    v_centers.append(point)
    v_radii.append(radius)

    max_pad = 0
    for pad in range(1, 40):
        a = max(0, edges[j + 1] - pad)
        b = min(len(w_curve) - 1, edges[j + 1] + pad)
        if np.all(np.abs(w_curve[a:b + 1] - point) <= radius):
            max_pad = pad
        else:
            break
    overlap_pads.append(max_pad)

print('Special-case chart diagnostics:')
for j in range(n_sets):
    print(
        f'  U_{j}: center={u_centers[j]:.6f}, radius={u_radii[j]:.6f}, '
        f'zero margin={abs(u_centers[j]) - u_radii[j]:.6f}'
    )
for j in range(n_sets - 1):
    print(
        f'  V_{j}: center={v_centers[j]:.6f}, radius={v_radii[j]:.6f}, '
        f'sample window pad={overlap_pads[j]}'
    )


## Explicit disk charts $U_j$ and overlap neighborhoods $V_j$

For this sandbox we choose actual open disks
$$
U_j = D(c_j, R_j)
$$
covering the four image segments, where $c_j$ is the mean of the sampled segment and $R_j$ is the maximal sampled distance to $c_j$ plus a small safety margin.

For each overlap point
$$
p_j = A_2(t_{j+1}),
$$
we then choose an actual open overlap disk
$$
V_j = D(p_j, \rho_j)
$$
with radius small enough that the whole disk sits inside both neighboring charts. This turns the phrase “construct $V_j$” into a concrete geometric operation.


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 6.3))
ax.plot(w_curve.real, w_curve.imag, color='black', lw=2.0, label=r'$A_2(\gamma_r)$')

for j, (center, radius, color) in enumerate(zip(u_centers, u_radii, colors)):
    disk = Circle((center.real, center.imag), radius, facecolor=color, edgecolor=color, alpha=0.12, lw=2)
    ax.add_patch(disk)
    ax.scatter([center.real], [center.imag], color=color, s=28, zorder=4)
    ax.text(center.real, center.imag, rf'$U_{j}$', color=color, fontsize=10, ha='center', va='center')

for j, (center, radius) in enumerate(zip(v_centers, v_radii)):
    disk = Circle((center.real, center.imag), radius, facecolor='none', edgecolor='crimson', linestyle='--', lw=2.2)
    ax.add_patch(disk)
    ax.scatter([center.real], [center.imag], color='crimson', s=26, zorder=5)
    ax.text(center.real + 0.05, center.imag + 0.06, rf'$V_{j}$', color='crimson', fontsize=10)

ax.axvline(0, color='0.75', lw=1.0, ls='--')
ax.text(0.04, 0.95, r'right half-plane', transform=ax.transAxes, color='darkgreen', fontsize=10)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(w_curve.real.min() - 0.45, w_curve.real.max() + 0.45)
ax.set_ylim(w_curve.imag.min() - 0.45, w_curve.imag.max() + 0.45)
ax.set_xlabel(r'$\Re w$')
ax.set_ylabel(r'$\Im w$')
ax.set_title(r'Special-case disk charts $U_j$ and overlap disks $V_j$ in the $w$-plane')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
plt.show()


## Special-case verification of the missing PLAN 09 input

The next calculation checks the exact ingredients that PLAN 09 wants from a special case:

1. each image segment is contained in its chosen chart $U_j$;
2. each overlap disk $V_j$ is contained in both neighboring charts;
3. a small time window around each overlap parameter maps into $V_j$;
4. on each $V_j$, the local logarithm branches $b_j$ and $b_{j+1}$ agree, and likewise the local root branches $\xi_j$ and $\xi_{j+1}$.

In this sandbox the branch-agreement step is easy because every chart lies in the right half-plane, so we can take the principal logarithm on every $U_j$ and restrict it. The general theorem has to reproduce this kind of agreement **without** assuming one global log domain for the whole loop.


In [ ]:
angles = np.linspace(0, 2 * np.pi, 256, endpoint=False)

print('Verification checks:')
for j, ((a, b), center, radius) in enumerate(zip(intervals, u_centers, u_radii)):
    seg = w_curve[a:b + 1]
    cover_ok = np.all(np.abs(seg - center) < radius)
    print(f'  segment {j} contained in U_{j}: {cover_ok}')

for j, (point, radius) in enumerate(zip(v_centers, v_radii)):
    boundary = point + 0.95 * radius * np.exp(1j * angles)
    left_ok = np.all(np.abs(boundary - u_centers[j]) < u_radii[j])
    right_ok = np.all(np.abs(boundary - u_centers[j + 1]) < u_radii[j + 1])

    pad = overlap_pads[j]
    a = max(0, edges[j + 1] - pad)
    b = min(len(w_curve) - 1, edges[j + 1] + pad)
    window_ok = np.all(np.abs(w_curve[a:b + 1] - point) < radius)

    sample = point + 0.6 * radius * np.exp(1j * angles)
    b_left = np.log(sample)
    b_right = np.log(sample)
    xi_left = np.exp(b_left / (2 ** K))
    xi_right = np.exp(b_right / (2 ** K))
    log_error = np.max(np.abs(b_left - b_right))
    root_error = np.max(np.abs(xi_left - xi_right))

    print(f'  V_{j} subset U_{j} ∩ U_{j + 1}: {left_ok and right_ok}')
    print(f'    sampled overlap window inside V_{j}: {window_ok}')
    print(f'    max log-branch mismatch on sampled points:  {log_error:.3e}')
    print(f'    max root-branch mismatch on sampled points: {root_error:.3e}')


## Proof outline suggested by this sandbox

The special case suggests the following proof template for PLAN 09.

1. Choose a high level and a partition so that each image segment is controlled by one local chart.
2. Produce actual zero-free simply connected charts $U_j$ around the controlled image segments.
3. Around each overlap value $A_K(t_{j+1})$, shrink to an explicit open neighborhood $V_j \subset U_j \cap U_{j+1}$.
4. Define the actual local logarithm branches by continuation of the normalized starting germ.
5. Prove that the neighboring branches agree on $V_j$, ideally by uniqueness of analytic continuation once both branches are known to continue the same germ there.
6. Package those data into the Lean interface
   ```lean
   HighEscapingActualChartChainsEventuallyEqAtOverlapsData
   ```
   and then invoke the already-formalized comparison theorem.

This notebook should therefore be treated as a geometric sandbox for **constructing $V_j$ first**, not as another notebook about the already-solved abstract monodromy step.


## Lean handoff target

The concrete output that PLAN 09 wants from a successful special-case argument is a package of the form

```lean
HighEscapingActualChartChainsEventuallyEqAtOverlapsData
```

built from explicit charts $U_j$, explicit overlap neighborhoods $V_j$, and actual proofs of branch equality on the $V_j$.

Once that package exists, the rest is already checked:

```lean
ChartChainLocalLogsEventuallyEqAtOverlaps.of_open_eqOn
BasinLoopChartChain.monodromyProduct_eq_one_of_open_eqOn
HighEscapingActualChartChainsEventuallyEqAtOverlapsData.toProductComparisonData
```

So the frontier is no longer the algebra of monodromy products. The frontier is the local geometric construction of the overlap neighborhoods themselves.
